# HW0. Your first TPU run

This Jupyter notebook has one Cloud TPU v5e chip attached for the entire session.
Run the cells directly; no job submission or queue is needed. A chip remains allocated
while your notebook server is running, even when you are only reading or editing.
Stop your server when you finish. Inactive servers are stopped automatically after
30 minutes, and every server has an eight-hour maximum age.


In [ ]:
import jax

devices = jax.devices()
print("devices:", devices)
assert devices and all(device.platform == "tpu" for device in devices), devices


You should see one `TpuDevice`. If this cell fails, tell a TA before continuing.

## Part 1. Attention, and where the time goes

Below is a single multi-head attention forward pass, the shape that dominates an LLM
forward pass. Run it, then answer the questions underneath.


In [ ]:
import time
import jax.numpy as jnp

B, H, S, D = 8, 12, 1024, 64
key = jax.random.PRNGKey(0)
q, k, v = jax.random.normal(key, (3, B, H, S, D), dtype=jnp.bfloat16)

@jax.jit
def attn(q, k, v):
    logits = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(D).astype(q.dtype)
    return jnp.einsum("bhqk,bhkd->bhqd", jax.nn.softmax(logits, axis=-1), v)

attn(q, k, v).block_until_ready()          # compile first, time second
t0 = time.perf_counter()
for _ in range(50):
    out = attn(q, k, v)
out.block_until_ready()
dt = (time.perf_counter() - t0) / 50

flops = 4 * B * H * S * S * D
print(f"{dt*1e3:.3f} ms per pass")
print(f"{flops/dt/1e12:.2f} TFLOP/s")


**Q1.** A v5e chip peaks near 197 TFLOP/s in bfloat16. You measured far less than that.
Work out how many bytes the `logits` array occupies and how many bytes the chip must
move to compute the softmax over it. Is this kernel limited by arithmetic or by memory
bandwidth? Show the arithmetic intensity you calculated.

**Q2.** Double `S` to 2048 and run again. Predict the change in time *before* you run
it, then explain any gap between your prediction and the measurement.

## Part 2. Make it faster

The two-einsum version above materialises the whole `B × H × S × S` logits array in
memory. Flash attention avoids that by tiling over the sequence and keeping a running
softmax, so the large intermediate never exists.

Use `jax.nn.dot_product_attention` and compare. Report both numbers and explain the
difference in terms of the bytes moved, not just the wall time.

In [ ]:
# Your code here. It runs directly on the TPU attached to this notebook.


## Part 3. Inspect a TPU trace with XProf

JAX records the trace; XProf displays what ran on the host and TPU. Run the
cell below to compile a matrix multiplication, then capture repeated TPU
executions. The trace is stored on your persistent home volume.


In [ ]:
import os
from html import escape
from pathlib import Path
from IPython.display import HTML, display
import jax
import jax.numpy as jnp

@jax.jit
def profile_matmul(x):
    return x @ x

profile_input = jnp.ones((2048, 2048), dtype=jnp.bfloat16)
profile_matmul(profile_input).block_until_ready()  # compile before tracing

profile_dir = Path.home() / "xprof"
with jax.profiler.trace(str(profile_dir)):
    for _ in range(20):
        profile_matmul(profile_input).block_until_ready()

public_origin = os.environ.get("HUB_PUBLIC_ORIGIN", "").rstrip("/")
service_prefix = os.environ["JUPYTERHUB_SERVICE_PREFIX"].rstrip("/")
xprof_url = f"{public_origin}{service_prefix}/xprof/"
display(HTML(
    f'<a href="{escape(xprof_url, quote=True)}" target="_blank" rel="noopener noreferrer">'
    'Open XProf</a>'
))


Click **Open XProf** above, or choose **XProf** from the JupyterLab launcher.
In the **Profile** tab, select the newest session
and choose **Trace Viewer** from the Tools menu. Look for the TPU and host tracks.
Each run adds another session.

For help reading the timeline, use the [XProf Trace Viewer guide](https://openxla.org/xprof/trace_viewer).
The [JAX profiling guide](https://docs.jax.dev/en/latest/profiling.html#xprof-tensorboard-profiling)
explains trace capture, annotations, and profiler options.


### Example: compare two attention sizes

Copy this into a new cell and run it with `profile_S = 1024`. Then change
`profile_S` to `2048` and run it again. Select each new session in XProf
and compare the TPU **XLA Ops**
track. The labelled event on the host track identifies your run. The warm-up
call keeps compilation outside the recorded region.

```python
profile_S = 1024  # repeat with 2048
profile_q, profile_k, profile_v = jax.random.normal(
    jax.random.PRNGKey(profile_S),
    (3, B, H, profile_S, D),
    dtype=jnp.bfloat16,
)
attn(profile_q, profile_k, profile_v).block_until_ready()
with jax.profiler.trace(str(profile_dir)):
    for _ in range(10):
        with jax.profiler.TraceAnnotation(f"attention S={profile_S}"):
            attn(profile_q, profile_k, profile_v).block_until_ready()
```


### Example: change trace detail

The first trace used JAX's default capture settings. For more host detail,
run the code below with `host_level = 3`. Use `host_level = 2` for the
default host detail, or set `python_level = 1` to include Python calls.
Keep this capture short because extra detail creates a larger trace.
Compare the host tracks with the first session; the TPU tracks remain enabled.

```python
host_level = 3    # 2 is the default
python_level = 0  # 1 also records Python calls
options = jax.profiler.ProfileOptions()
options.host_tracer_level = host_level
options.python_tracer_level = python_level
with jax.profiler.trace(str(profile_dir), profiler_options=options):
    for _ in range(5):
        profile_matmul(profile_input).block_until_ready()
```


## Finishing your session

Save your notebook with its outputs, then click the red Stop TPU Server button
in JupyterLab. Closing the browser tab alone may leave the TPU allocated until the idle
culler stops the server.

## Submitting

Hand in this notebook with all outputs kept. The first cell records the TPU device
used for the measurements.
